# Viettel AI Race - Distill model từ bản 82x

Notebook này train NER bằng pseudo-label từ `82x_cleanroom4_k2` (bản public 38.6460), rồi chạy inference bằng model vừa train. Ý tưởng là để model học lại cấu trúc nhãn tốt hơn bản tay cũ, không nộp nguyên output teacher.

## 1. Upload gói data

Ở máy local chạy `bash colab/pack.sh`, rồi upload `artifacts/viettel_colab_data.zip` ở cell dưới.

In [ ]:
from google.colab import files
from pathlib import Path
import os, shutil, zipfile

uploaded = files.upload()
zip_names = [name for name in uploaded if name.endswith('.zip')]
assert zip_names, 'Bạn cần upload artifacts/viettel_colab_data.zip'

work = Path('/content/vietel_distill')
if work.exists():
    shutil.rmtree(work)
work.mkdir(parents=True)

with zipfile.ZipFile(zip_names[0]) as zf:
    zf.extractall(work)
os.chdir(work)
print('Đã giải nén vào', work)
!find data -maxdepth 2 -type d | sort
!wc -l data/ner_teacher_82x/*.jsonl

## 2. Cài thư viện và kiểm GPU

In [ ]:
!pip install -q -r requirements.txt

import torch
print('cuda:', torch.cuda.is_available())
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))

MODEL_NAME = 'xlm-roberta-large'
EPOCHS_DEV = 20
EPOCHS_FINAL = 30
BS = 4
ACCUM = 2
LR = 2e-5
PRECISION_FLAG = '--bf16' if torch.cuda.is_available() and torch.cuda.is_bf16_supported() else ('--fp16' if torch.cuda.is_available() else '')
print('precision:', PRECISION_FLAG or 'fp32')

## 3. Build lại data teacher

In [ ]:
!python src/gt_lexicon.py --gt data/teacher_82x --out data/kb/gt_lexicon_teacher_82x.json --split all
!python src/ner_data.py --gt data/teacher_82x --out data/ner_teacher_82x

## 4. Train dev để xem model học teacher tới đâu

In [ ]:
!python src/ner_train.py --data data/ner_teacher_82x --model $MODEL_NAME --out models/ner_teacher_dev --epochs $EPOCHS_DEV --bs $BS --accum $ACCUM --lr $LR $PRECISION_FLAG 2>&1 | tee /content/train_teacher_dev.log

## 5. Train final trên toàn bộ 100 file

In [ ]:
!python src/ner_train.py --data data/ner_teacher_82x --all --model $MODEL_NAME --out models/ner_teacher_82x --epochs $EPOCHS_FINAL --bs $BS --accum $ACCUM --lr $LR $PRECISION_FLAG 2>&1 | tee /content/train_teacher_final.log

## 6. Sinh output bằng model distill

In [ ]:
!rm -rf /content/pred_teacher_82x /content/output
!python src/ner_infer.py --model models/ner_teacher_82x --input input --out /content/pred_teacher_82x --lexicon data/kb/gt_lexicon_teacher_82x.json

## 7. Kiểm offset và tải zip nộp

In [ ]:
import json, shutil
from pathlib import Path

bad = []
for fp in sorted(Path('/content/pred_teacher_82x').glob('*.json'), key=lambda p: int(p.stem)):
    raw = Path('input', f'{fp.stem}.txt').read_text(encoding='utf-8')
    ents = json.loads(fp.read_text(encoding='utf-8'))
    for i, e in enumerate(ents):
        s, t = e['position']
        if raw[s:t] != e['text']:
            bad.append((fp.name, i, e['position'], e['text'], raw[s:t]))
assert not bad, bad[:3]
print('Offset OK:', len(list(Path('/content/pred_teacher_82x').glob('*.json'))), 'files')

!rm -rf /content/output
!mkdir -p /content/output
!cp /content/pred_teacher_82x/*.json /content/output/
!cd /content && zip -q -r output_teacher_model_82x.zip output
!ls -lh /content/output_teacher_model_82x.zip
files.download('/content/output_teacher_model_82x.zip')

## 8. Probe copy field từ teacher

Hai zip này không thêm/bớt entity. Chúng chỉ copy `candidates/assertions` khi model đã bắt đúng tuyệt đối cùng position và type.

In [ ]:
!python src/compare_to_teacher.py --pred /content/pred_teacher_82x --teacher data/teacher_82x

!python src/copy_teacher_fields.py --pred /content/pred_teacher_82x --teacher data/teacher_82x --input input --out /content/pred_teacher_82x_fields82
!rm -rf /content/output
!mkdir -p /content/output
!cp /content/pred_teacher_82x_fields82/*.json /content/output/
!cd /content && zip -q -r output_teacher_model_fields82.zip output

!python src/copy_teacher_fields.py --pred /content/pred_teacher_82x --teacher data/teacher_85 --input input --out /content/pred_teacher_82x_fields85
!rm -rf /content/output
!mkdir -p /content/output
!cp /content/pred_teacher_82x_fields85/*.json /content/output/
!cd /content && zip -q -r output_teacher_model_fields85.zip output

!ls -lh /content/output_teacher_model_82x.zip /content/output_teacher_model_fields82.zip /content/output_teacher_model_fields85.zip
files.download('/content/output_teacher_model_fields82.zip')
files.download('/content/output_teacher_model_fields85.zip')

## 9. Tải weights để lưu lại run

In [ ]:
!cd models && zip -q -r /content/ner_teacher_82x_weights.zip ner_teacher_82x
!ls -lh /content/ner_teacher_82x_weights.zip
files.download('/content/ner_teacher_82x_weights.zip')
files.download('/content/train_teacher_dev.log')
files.download('/content/train_teacher_final.log')